In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from joblib import dump
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("=== 肿瘤抗原预测模型训练（使用CNN和新特征集） ===")

class TumorFeatureExtractor:
    def __init__(self, acid_file_path='acid.xlsx'):
        # 加载氨基酸物化性质表
        self.acid_properties = self._load_acid_properties(acid_file_path)
        print(f"成功加载 {len(self.acid_properties)} 种氨基酸的物化性质")
        
        # 氨基酸编码字典
        self.amino_acid_encoding = {
            'A': 1, 'R': 2, 'N': 3, 'D': 4, 'C': 5, 'Q': 6, 'E': 7, 'G': 8, 'H': 9, 'I': 10,
            'L': 11, 'K': 12, 'M': 13, 'F': 14, 'P': 15, 'S': 16, 'T': 17, 'W': 18, 'Y': 19, 'V': 20
        }
        
        # 定义新特征名称
        self.feature_names = [
            "sequence_length",
            "avg_Hydrophobicity", "std_Hydrophobicity", "total_Hydrophobicity",
            "avg_pI", "std_pI", "total_pI",
            "avg_MW", "std_MW", "total_MW",
            "avg_ele", "std_ele", "total_ele",
            "avg_plo", "std_plo", "total_plo",
            "avg_Hydrocapto", "std_Hydrocapto", "total_Hydrocapto",
            "A_composition", "R_composition", "N_composition", "D_composition", "C_composition",
            "Q_composition", "E_composition", "G_composition", "H_composition", "I_composition",
            "L_composition", "K_composition", "M_composition", "F_composition", "P_composition",
            "S_composition", "T_composition", "W_composition", "Y_composition", "V_composition",
            "P2_L", "P3_L_neg", "P4_LVI", "P9_L", "P9_V", "P6_L", "P7-8_LVI_neg",
            "AA1", "AA2", "AA3", "AA4", "AA5", "AA6", "AA7", "AA8", "AA9"
        ]
        
        # 添加位置相关的物理化学特征
        for i in range(1, 10):  # P1到P9
            self.feature_names.extend([
                f"P{i}_Hydrophobicity", f"P{i}_pI", f"P{i}_MW", 
                f"P{i}_ele", f"P{i}_plo", f"P{i}_Hydrocapto"
            ])
    
    def _load_acid_properties(self, file_path):
        try:
            df = pd.read_excel(file_path)
            df.columns = [col.strip() for col in df.columns]
            df['acid'] = df['acid'].astype(str).str.strip()
            
            properties_dict = {}
            for _, row in df.iterrows():
                acid = row['acid']
                properties_dict[acid] = {
                    'Hydrophobicity': row['Hydrophobicity'],
                    'pI': row['pI'],
                    'MW': row['MW'],
                    'ele': row['ele'],
                    'plo': row['plo'],
                    'Hydrocapto': row['Hydrocapto']
                }
            return properties_dict
        except Exception as e:
            print(f"加载氨基酸性质文件失败: {e}")
            return {}
    
    def extract_features(self, sequences):
        if not sequences:
            raise ValueError("序列列表为空")
        
        print(f"正在为 {len(sequences)} 个序列提取新特征集...")
        features_list = []
        
        for i, seq in enumerate(sequences):
            if i % 1000 == 0 and i > 0:
                print(f"已处理 {i}/{len(sequences)} 个序列")
            
            features = self._extract_single_sequence_features(seq)
            features_list.append(features)
        
        feature_df = pd.DataFrame(features_list, columns=self.feature_names)
        
        print(f"特征提取完成，共生成 {len(self.feature_names)} 个特征")
        return feature_df
    
    def _extract_single_sequence_features(self, sequence):
        """生成新特征向量"""
        features = []
        
        # 1. 序列长度
        features.append(len(sequence))
        
        # 2. 全局物理化学特征统计（平均值、标准差、总和）
        physicochemical_stats = self._calculate_global_physicochemical_stats(sequence)
        features.extend(physicochemical_stats)
        
        # 3. 氨基酸组成
        aa_composition = self._calculate_amino_acid_composition(sequence)
        features.extend(aa_composition)
        
        # 4. 生物学规则特征
        rule_features = self._calculate_rule_features(sequence)
        features.extend(rule_features)
        
        # 5. 氨基酸编码（前9个位置）
        aa_encoding = self._calculate_amino_acid_encoding(sequence)
        features.extend(aa_encoding)
        
        # 6. 位置相关的物理化学特征
        positional_features = self._calculate_positional_physicochemical_features(sequence)
        features.extend(positional_features)
        
        return features
    
    def _calculate_global_physicochemical_stats(self, sequence):
        """计算全局物理化学特征统计"""
        stats = []
        properties_list = ['Hydrophobicity', 'pI', 'MW', 'ele', 'plo', 'Hydrocapto']
        
        for prop in properties_list:
            values = []
            for aa in sequence:
                aa_upper = aa.upper()
                if aa_upper in self.acid_properties:
                    values.append(self.acid_properties[aa_upper][prop])
            
            if values:
                stats.extend([np.mean(values), np.std(values), np.sum(values)])
            else:
                stats.extend([0.0, 0.0, 0.0])
        
        return stats
    
    def _calculate_amino_acid_composition(self, sequence):
        """计算20种氨基酸的组成比例"""
        composition = np.zeros(20)
        aa_list = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
                  'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
        
        if sequence:
            total_aa = len(sequence)
            for aa in sequence:
                aa_upper = aa.upper()
                if aa_upper in aa_list:
                    idx = aa_list.index(aa_upper)
                    composition[idx] += 1
            
            # 转换为比例
            composition = composition / total_aa
        
        return composition.tolist()
    
    def _calculate_rule_features(self, sequence):
        """计算7个生物学规则特征"""
        rules = np.zeros(7)
        if len(sequence) > 1 and sequence[1].upper() == 'L': 
            rules[0] = 1
        if len(sequence) > 2 and sequence[2].upper() == 'L': 
            rules[1] = 1
        if len(sequence) > 3 and sequence[3].upper() in {'L', 'V', 'I'}: 
            rules[2] = 1
        if len(sequence) > 8 and sequence[8].upper() == 'L': 
            rules[3] = 1
        if len(sequence) > 8 and sequence[8].upper() == 'V': 
            rules[4] = 1
        if len(sequence) > 5 and sequence[5].upper() == 'L': 
            rules[5] = 1
        if (len(sequence) > 6 and sequence[6].upper() in {'L', 'V', 'I'}) or \
           (len(sequence) > 7 and sequence[7].upper() in {'L', 'V', 'I'}): 
            rules[6] = 1
        return rules.tolist()
    
    def _calculate_amino_acid_encoding(self, sequence):
        """计算前9个位置的氨基酸编码"""
        max_length = 9
        encoded = [self.amino_acid_encoding.get(aa.upper(), 0) for aa in sequence[:max_length]]
        encoded += [0] * (max_length - len(encoded))
        return encoded
    
    def _calculate_positional_physicochemical_features(self, sequence):
        """计算位置相关的物理化学特征（P1-P9）"""
        features = []
        max_length = 9
        
        for i in range(max_length):
            if i < len(sequence):
                aa = sequence[i].upper()
                if aa in self.acid_properties:
                    features.extend([
                        self.acid_properties[aa]['Hydrophobicity'],
                        self.acid_properties[aa]['pI'],
                        self.acid_properties[aa]['MW'],
                        self.acid_properties[aa]['ele'],
                        self.acid_properties[aa]['plo'],
                        self.acid_properties[aa]['Hydrocapto']
                    ])
                else:
                    features.extend([0.0]*6)  # 未知氨基酸用0填充
            else:
                features.extend([0.0]*6)  # 不足位置用0填充
        return features
    
    def get_feature_names(self):
        return self.feature_names

def build_cnn_model(input_shape=(109, 1)):
    """构建CNN模型 - 调整输入形状为109个特征"""
    model = Sequential([
        # 第一卷积块
        Conv1D(64, 3, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        
        # 第二卷积块
        Conv1D(64, 3, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        
        # 分类头
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    
    return model

def main():
    # 检查文件
    if not os.path.exists('acid.xlsx'):
        print("错误: 未找到 acid.xlsx 文件")
        return
    
    if not os.path.exists('A0201_160000.xlsx'):
        print("错误: 未找到 A0201_160000.xlsx 文件")
        return
    
    try:
        # 1. 加载数据
        print("1. 加载训练数据...")
        dataset = pd.read_excel('A0201_160000.xlsx', na_filter=False)
        print(f"成功加载 {len(dataset)} 条序列")
        print(f"正样本: {sum(dataset['label'])}, 负样本: {len(dataset) - sum(dataset['label'])}")
        
        # 2. 提取特征
        print("2. 提取新特征集...")
        extractor = TumorFeatureExtractor('acid.xlsx')
        feature_df = extractor.extract_features(dataset['sequence'].tolist())
        X = feature_df.values
        y = dataset['label'].values
        
        print(f"特征矩阵形状: {X.shape}")
        
        # 3. 数据划分
        print("\n3. 划分训练集和测试集...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
        
        # 4. 特征标准化
        print("4. 特征标准化...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # 为CNN准备数据 - 重塑为3D格式 (样本数, 特征数, 1)
        X_train_cnn = X_train_scaled.reshape(-1, X_train_scaled.shape[1], 1)
        X_test_cnn = X_test_scaled.reshape(-1, X_test_scaled.shape[1], 1)
        
        print(f"CNN训练数据形状: {X_train_cnn.shape}")
        print(f"CNN测试数据形状: {X_test_cnn.shape}")
        
        # 5. 构建和训练CNN模型
        print("5. 构建和训练CNN模型...")
        model = build_cnn_model(input_shape=(X_train_cnn.shape[1], 1))
        
        # 打印模型结构
        model.summary()
        
        # 编译模型
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
        )
        
        # 设置回调函数
        callbacks = [
            EarlyStopping(
                monitor='val_auc',
                patience=20,
                mode='max',
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=10,
                min_lr=1e-7,
                verbose=1
            )
        ]
        
        # 计算类别权重以处理不平衡数据
        class_weight = {
            0: 1.0,
            1: len(y_train[y_train==0]) / len(y_train[y_train==1])
        }
        
        # 训练模型
        history = model.fit(
            X_train_cnn, y_train,
            validation_data=(X_test_cnn, y_test),
            epochs=150,
            batch_size=64,
            callbacks=callbacks,
            class_weight=class_weight,
            verbose=1
        )
        
        # 6. 评估模型
        print("6. 模型评估...")
        y_pred_prob = model.predict(X_test_cnn)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()
        
        accuracy = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_pred_prob)
        
        print(f"测试准确率: {accuracy:.4f}")
        print(f"测试AUC: {auc:.4f}")
        
        print("\n分类报告:")
        print(classification_report(y_test, y_pred))
        
        print("\n混淆矩阵:")
        print(confusion_matrix(y_test, y_pred))
        
        # 7. 保存模型和预处理对象
        print("7. 保存模型...")
        # 保存Keras模型
        model.save('tumor_model_new_features_cnn.h5')
        # 保存标准化器
        dump(scaler, 'tumor_scaler_new_features.joblib')
        
        # 保存所有特征的索引
        all_feature_indices = np.arange(len(extractor.feature_names))
        np.save('tumor_feature_indices_new_features.npy', all_feature_indices)
        
        # 保存特征名称
        with open('selected_feature_names_new_features.txt', 'w') as f:
            for name in extractor.feature_names:
                f.write(name + '\n')
        
        print("模型训练完成!")
        print(f"模型文件: tumor_model_new_features_cnn.h5")
        print(f"标准化器: tumor_scaler_new_features.joblib") 
        print(f"特征索引: tumor_feature_indices_new_features.npy")
        print(f"特征名称: selected_feature_names_new_features.txt")
        print(f"使用的特征数量: {len(all_feature_indices)}")
        
        # 显示特征类别统计
        feature_categories = {
            "序列基本信息": ["sequence_length"],
            "全局物化统计": [
                "avg_Hydrophobicity", "std_Hydrophobicity", "total_Hydrophobicity",
                "avg_pI", "std_pI", "total_pI", "avg_MW", "std_MW", "total_MW",
                "avg_ele", "std_ele", "total_ele", "avg_plo", "std_plo", "total_plo",
                "avg_Hydrocapto", "std_Hydrocapto", "total_Hydrocapto"
            ],
            "氨基酸组成": [f"{aa}_composition" for aa in ['A','R','N','D','C','Q','E','G','H','I','L','K','M','F','P','S','T','W','Y','V']],
            "规则特征": ["P2_L", "P3_L_neg", "P4_LVI", "P9_L", "P9_V", "P6_L", "P7-8_LVI_neg"],
            "氨基酸编码": [f"AA{i}" for i in range(1, 10)],
            "位置物化特征": [f"P{i}_{prop}" for i in range(1, 10) for prop in ['Hydrophobicity', 'pI', 'MW', 'ele', 'plo', 'Hydrocapto']]
        }
        
        print("\n=== 特征类别统计 ===")
        total_features = 0
        for category, features in feature_categories.items():
            print(f"{category}: {len(features)} 个特征")
            total_features += len(features)
        print(f"总计: {total_features} 个特征")
        
        # 输出训练历史中的最佳值
        best_epoch = np.argmax(history.history['val_auc'])
        print(f"\n最佳epoch: {best_epoch + 1}")
        print(f"最佳验证AUC: {history.history['val_auc'][best_epoch]:.4f}")
        print(f"最佳验证准确率: {history.history['val_accuracy'][best_epoch]:.4f}")
        
    except Exception as e:
        print(f"训练过程中出错: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()


=== 肿瘤抗原预测模型训练（使用CNN和新特征集） ===
1. 加载训练数据...
成功加载 167642 条序列
正样本: 50308, 负样本: 117334
2. 提取新特征集...
成功加载 20 种氨基酸的物化性质
正在为 167642 个序列提取新特征集...
已处理 1000/167642 个序列
已处理 2000/167642 个序列
已处理 3000/167642 个序列
已处理 4000/167642 个序列
已处理 5000/167642 个序列
已处理 6000/167642 个序列
已处理 7000/167642 个序列
已处理 8000/167642 个序列
已处理 9000/167642 个序列
已处理 10000/167642 个序列
已处理 11000/167642 个序列
已处理 12000/167642 个序列
已处理 13000/167642 个序列
已处理 14000/167642 个序列
已处理 15000/167642 个序列
已处理 16000/167642 个序列
已处理 17000/167642 个序列
已处理 18000/167642 个序列
已处理 19000/167642 个序列
已处理 20000/167642 个序列
已处理 21000/167642 个序列
已处理 22000/167642 个序列
已处理 23000/167642 个序列
已处理 24000/167642 个序列
已处理 25000/167642 个序列
已处理 26000/167642 个序列
已处理 27000/167642 个序列
已处理 28000/167642 个序列
已处理 29000/167642 个序列
已处理 30000/167642 个序列
已处理 31000/167642 个序列
已处理 32000/167642 个序列
已处理 33000/167642 个序列
已处理 34000/167642 个序列
已处理 35000/167642 个序列
已处理 36000/167642 个序列
已处理 37000/167642 个序列
已处理 38000/167642 个序列
已处理 39000/167642 个序列
已处理 40000/167642 个序列
已处理 41000/167642 个序列
已处理 4200

Epoch 6/150
2096/2096 [==============================] - 34s 16ms/step - loss: 0.3134 - accuracy: 0.9167 - auc: 0.9692 - val_loss: 0.1976 - val_accuracy: 0.9253 - val_auc: 0.9735 - lr: 0.0010
Epoch 7/150
2096/2096 [==============================] - 34s 16ms/step - loss: 0.3088 - accuracy: 0.9175 - auc: 0.9701 - val_loss: 0.1938 - val_accuracy: 0.9271 - val_auc: 0.9746 - lr: 0.0010
Epoch 8/150
2096/2096 [==============================] - 34s 16ms/step - loss: 0.3037 - accuracy: 0.9194 - auc: 0.9710 - val_loss: 0.2054 - val_accuracy: 0.9202 - val_auc: 0.9748 - lr: 0.0010
Epoch 9/150
2096/2096 [==============================] - 34s 16ms/step - loss: 0.2989 - accuracy: 0.9210 - auc: 0.9720 - val_loss: 0.1903 - val_accuracy: 0.9263 - val_auc: 0.9751 - lr: 0.0010
Epoch 10/150
2096/2096 [==============================] - 34s 16ms/step - loss: 0.2968 - accuracy: 0.9217 - auc: 0.9724 - val_loss: 0.2065 - val_accuracy: 0.9213 - val_auc: 0.9753 - lr: 0.0010
Epoch 11/150
2096/2096 [===============

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



2096/2096 [==============================] - 48s 23ms/step - loss: 0.2424 - accuracy: 0.9343 - auc: 0.9815 - val_loss: 0.1866 - val_accuracy: 0.9286 - val_auc: 0.9764 - lr: 5.0000e-04
Epoch 35/150
2096/2096 [==============================] - 49s 23ms/step - loss: 0.2427 - accuracy: 0.9331 - auc: 0.9815 - val_loss: 0.1943 - val_accuracy: 0.9248 - val_auc: 0.9763 - lr: 5.0000e-04
Epoch 36/150
2096/2096 [==============================] - 48s 23ms/step - loss: 0.2439 - accuracy: 0.9320 - auc: 0.9814 - val_loss: 0.1891 - val_accuracy: 0.9277 - val_auc: 0.9764 - lr: 5.0000e-04
Epoch 37/150
2096/2096 [==============================] - 48s 23ms/step - loss: 0.2404 - accuracy: 0.9333 - auc: 0.9818 - val_loss: 0.1918 - val_accuracy: 0.9259 - val_auc: 0.9764 - lr: 5.0000e-04
Epoch 38/150
 675/2096 [========>.....................] - ETA: 29s - loss: 0.2450 - accuracy: 0.9327 - auc: 0.9814

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



2096/2096 [==============================] - 51s 24ms/step - loss: 0.2345 - accuracy: 0.9348 - auc: 0.9828 - val_loss: 0.1923 - val_accuracy: 0.9266 - val_auc: 0.9763 - lr: 2.5000e-04
Epoch 44/150
2096/2096 [==============================] - 51s 24ms/step - loss: 0.2327 - accuracy: 0.9348 - auc: 0.9830 - val_loss: 0.1944 - val_accuracy: 0.9257 - val_auc: 0.9765 - lr: 2.5000e-04
Epoch 45/150
2096/2096 [==============================] - 48s 23ms/step - loss: 0.2319 - accuracy: 0.9347 - auc: 0.9831 - val_loss: 0.1894 - val_accuracy: 0.9279 - val_auc: 0.9763 - lr: 2.5000e-04
Epoch 46/150
2096/2096 [==============================] - 48s 23ms/step - loss: 0.2328 - accuracy: 0.9352 - auc: 0.9830 - val_loss: 0.1944 - val_accuracy: 0.9259 - val_auc: 0.9763 - lr: 2.5000e-04
Epoch 47/150
2030/2096 [============================>.] - ETA: 1s - loss: 0.2314 - accuracy: 0.9354 - auc: 0.9832

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [6]:
#Prediction
import pandas as pd
import numpy as np
from joblib import load
import os
import tensorflow as tf
from tensorflow.keras.models import load_model

print("=== 肿瘤抗原预测（使用CNN和新特征集） ===")

class TumorFeatureExtractor:
    def __init__(self, acid_file_path='acid.xlsx'):
        # 加载氨基酸物化性质表
        self.acid_properties = self._load_acid_properties(acid_file_path)
        print(f"成功加载 {len(self.acid_properties)} 种氨基酸的物化性质")
        
        # 氨基酸编码字典
        self.amino_acid_encoding = {
            'A': 1, 'R': 2, 'N': 3, 'D': 4, 'C': 5, 'Q': 6, 'E': 7, 'G': 8, 'H': 9, 'I': 10,
            'L': 11, 'K': 12, 'M': 13, 'F': 14, 'P': 15, 'S': 16, 'T': 17, 'W': 18, 'Y': 19, 'V': 20
        }
        
        # 定义新特征名称
        self.feature_names = [
            "sequence_length",
            "avg_Hydrophobicity", "std_Hydrophobicity", "total_Hydrophobicity",
            "avg_pI", "std_pI", "total_pI",
            "avg_MW", "std_MW", "total_MW",
            "avg_ele", "std_ele", "total_ele",
            "avg_plo", "std_plo", "total_plo",
            "avg_Hydrocapto", "std_Hydrocapto", "total_Hydrocapto",
            "A_composition", "R_composition", "N_composition", "D_composition", "C_composition",
            "Q_composition", "E_composition", "G_composition", "H_composition", "I_composition",
            "L_composition", "K_composition", "M_composition", "F_composition", "P_composition",
            "S_composition", "T_composition", "W_composition", "Y_composition", "V_composition",
            "P2_L", "P3_L_neg", "P4_LVI", "P9_L", "P9_V", "P6_L", "P7-8_LVI_neg",
            "AA1", "AA2", "AA3", "AA4", "AA5", "AA6", "AA7", "AA8", "AA9"
        ]
        
        # 添加位置相关的物理化学特征
        for i in range(1, 10):  # P1到P9
            self.feature_names.extend([
                f"P{i}_Hydrophobicity", f"P{i}_pI", f"P{i}_MW", 
                f"P{i}_ele", f"P{i}_plo", f"P{i}_Hydrocapto"
            ])
    
    def _load_acid_properties(self, file_path):
        try:
            df = pd.read_excel(file_path)
            df.columns = [col.strip() for col in df.columns]
            df['acid'] = df['acid'].astype(str).str.strip()
            
            properties_dict = {}
            for _, row in df.iterrows():
                acid = row['acid']
                properties_dict[acid] = {
                    'Hydrophobicity': row['Hydrophobicity'],
                    'pI': row['pI'],
                    'MW': row['MW'],
                    'ele': row['ele'],
                    'plo': row['plo'],
                    'Hydrocapto': row['Hydrocapto']
                }
            return properties_dict
        except Exception as e:
            print(f"加载氨基酸性质文件失败: {e}")
            return {}
    
    def extract_features(self, sequences):
        if not sequences:
            raise ValueError("序列列表为空")
        
        print(f"正在为 {len(sequences)} 个序列提取新特征集...")
        features_list = []
        
        for i, seq in enumerate(sequences):
            if i % 1000 == 0 and i > 0:
                print(f"已处理 {i}/{len(sequences)} 个序列")
            
            features = self._extract_single_sequence_features(seq)
            features_list.append(features)
        
        feature_df = pd.DataFrame(features_list, columns=self.feature_names)
        
        print(f"特征提取完成，共生成 {len(self.feature_names)} 个特征")
        return feature_df
    
    def _extract_single_sequence_features(self, sequence):
        """生成新特征向量"""
        features = []
        
        # 1. 序列长度
        features.append(len(sequence))
        
        # 2. 全局物理化学特征统计（平均值、标准差、总和）
        physicochemical_stats = self._calculate_global_physicochemical_stats(sequence)
        features.extend(physicochemical_stats)
        
        # 3. 氨基酸组成
        aa_composition = self._calculate_amino_acid_composition(sequence)
        features.extend(aa_composition)
        
        # 4. 生物学规则特征
        rule_features = self._calculate_rule_features(sequence)
        features.extend(rule_features)
        
        # 5. 氨基酸编码（前9个位置）
        aa_encoding = self._calculate_amino_acid_encoding(sequence)
        features.extend(aa_encoding)
        
        # 6. 位置相关的物理化学特征
        positional_features = self._calculate_positional_physicochemical_features(sequence)
        features.extend(positional_features)
        
        return features
    
    def _calculate_global_physicochemical_stats(self, sequence):
        """计算全局物理化学特征统计"""
        stats = []
        properties_list = ['Hydrophobicity', 'pI', 'MW', 'ele', 'plo', 'Hydrocapto']
        
        for prop in properties_list:
            values = []
            for aa in sequence:
                aa_upper = aa.upper()
                if aa_upper in self.acid_properties:
                    values.append(self.acid_properties[aa_upper][prop])
            
            if values:
                stats.extend([np.mean(values), np.std(values), np.sum(values)])
            else:
                stats.extend([0.0, 0.0, 0.0])
        
        return stats
    
    def _calculate_amino_acid_composition(self, sequence):
        """计算20种氨基酸的组成比例"""
        composition = np.zeros(20)
        aa_list = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
                  'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
        
        if sequence:
            total_aa = len(sequence)
            for aa in sequence:
                aa_upper = aa.upper()
                if aa_upper in aa_list:
                    idx = aa_list.index(aa_upper)
                    composition[idx] += 1
            
            # 转换为比例
            composition = composition / total_aa
        
        return composition.tolist()
    
    def _calculate_rule_features(self, sequence):
        """计算7个生物学规则特征"""
        rules = np.zeros(7)
        if len(sequence) > 1 and sequence[1].upper() == 'L': 
            rules[0] = 1
        if len(sequence) > 2 and sequence[2].upper() == 'L': 
            rules[1] = 1
        if len(sequence) > 3 and sequence[3].upper() in {'L', 'V', 'I'}: 
            rules[2] = 1
        if len(sequence) > 8 and sequence[8].upper() == 'L': 
            rules[3] = 1
        if len(sequence) > 8 and sequence[8].upper() == 'V': 
            rules[4] = 1
        if len(sequence) > 5 and sequence[5].upper() == 'L': 
            rules[5] = 1
        if (len(sequence) > 6 and sequence[6].upper() in {'L', 'V', 'I'}) or \
           (len(sequence) > 7 and sequence[7].upper() in {'L', 'V', 'I'}): 
            rules[6] = 1
        return rules.tolist()
    
    def _calculate_amino_acid_encoding(self, sequence):
        """计算前9个位置的氨基酸编码"""
        max_length = 9
        encoded = [self.amino_acid_encoding.get(aa.upper(), 0) for aa in sequence[:max_length]]
        encoded += [0] * (max_length - len(encoded))
        return encoded
    
    def _calculate_positional_physicochemical_features(self, sequence):
        """计算位置相关的物理化学特征（P1-P9）"""
        features = []
        max_length = 9
        
        for i in range(max_length):
            if i < len(sequence):
                aa = sequence[i].upper()
                if aa in self.acid_properties:
                    features.extend([
                        self.acid_properties[aa]['Hydrophobicity'],
                        self.acid_properties[aa]['pI'],
                        self.acid_properties[aa]['MW'],
                        self.acid_properties[aa]['ele'],
                        self.acid_properties[aa]['plo'],
                        self.acid_properties[aa]['Hydrocapto']
                    ])
                else:
                    features.extend([0.0]*6)  # 未知氨基酸用0填充
            else:
                features.extend([0.0]*6)  # 不足位置用0填充
        return features
    
    def get_feature_names(self):
        return self.feature_names

def main():
    # 检查必要文件 - 使用新特征集的CNN模型文件
    required_files = ['tumor_feature_indices_new_features.npy', 'tumor_scaler_new_features.joblib', 'tumor_model_new_features_cnn.h5']
    missing = [f for f in required_files if not os.path.exists(f)]
    if missing:
        print(f"错误: 缺失模型文件 {missing}")
        print("请先运行训练代码训练新特征集CNN模型")
        return
    
    if not os.path.exists('acid.xlsx'):
        print("错误: 未找到 acid.xlsx 文件")
        return
    
    # 文件路径
    input_file = 'A0201_100000_test.xlsx'
    output_file = "A0201_100000_test.csv"
    
    try:
        # 1. 加载模型和预处理对象
        print("1. 加载新特征集CNN模型...")
        feature_indices = np.load('tumor_feature_indices_new_features.npy')
        scaler = load('tumor_scaler_new_features.joblib')
        model = load_model('tumor_model_new_features_cnn.h5')
        
        print(f"加载了 {len(feature_indices)} 个特征索引")
        
        # 2. 加载新数据
        print("2. 加载预测数据...")
        new_data = pd.read_excel(input_file)
        print(f"成功加载 {len(new_data)} 条序列")
        
        # 3. 提取新特征集
        print("3. 提取新特征集...")
        extractor = TumorFeatureExtractor('acid.xlsx')
        feature_df = extractor.extract_features(new_data['sequence'].tolist())
        X_new = feature_df.values
        
        print(f"提取的特征矩阵形状: {X_new.shape}")
        
        # 4. 应用特征标准化
        X_new_scaled = scaler.transform(X_new)
        
        # 为CNN准备数据 - 重塑为3D格式 (样本数, 特征数, 1)
        num_features = X_new_scaled.shape[1]
        X_new_cnn = X_new_scaled.reshape(-1, num_features, 1)
        
        print(f"CNN输入数据形状: {X_new_cnn.shape}")
        
        # 5. 预测
        print("4. 进行预测...")
        predictions_prob = model.predict(X_new_cnn).flatten()
        predictions_label = (predictions_prob > 0.5).astype(int)
        
        # 6. 保存结果
        print("5. 保存结果...")
        result_df = pd.DataFrame({
            'sequence': new_data['sequence'],
            'Prediction_Score': predictions_prob,
            'prediction': predictions_label
        })
        result_df.to_csv(output_file, index=False)
        
        # 7. 统计结果
        print("6. 预测结果统计:")
        total_samples = len(result_df)
        positive_count = result_df['prediction'].sum()
        positive_ratio = (positive_count / total_samples) * 100
        
        print(f"总样本数: {total_samples}")
        print(f"阳性预测数: {positive_count}")
        print(f"阳性预测比例: {positive_ratio:.2f}%")
        
        # 8. 显示预测分数分布
        print("\n预测分数分布:")
        score_ranges = [(0.9, 1.0), (0.8, 0.9), (0.7, 0.8), (0.6, 0.7), (0.5, 0.6), (0.0, 0.5)]
        for min_score, max_score in score_ranges:
            if max_score == 1.0:
                count = (predictions_prob >= min_score).sum()
            else:
                count = ((predictions_prob >= min_score) & (predictions_prob < max_score)).sum()
            percentage = (count / total_samples) * 100
            print(f"分数 {min_score:.1f}-{max_score:.1f}: {count} 条序列 ({percentage:.1f}%)")
        
        # 9. 显示top预测结果
        print("\n7. Top 10 高置信度预测:")
        top_predictions = result_df.nlargest(10, 'Prediction_Score')[['sequence', 'Prediction_Score', 'prediction']]
        for idx, row in top_predictions.iterrows():
            score = row['Prediction_Score']
            label = row['prediction']
            print(f"{row['sequence']} -> 得分: {score:.4f}, 预测: {'阳性' if label == 1 else '阴性'}")
        
        # 10. 显示阳性预测结果
        positive_results = result_df[result_df['prediction'] == 1]
        if len(positive_results) > 0:
            print(f"\n8. 所有阳性预测结果 (共{len(positive_results)}条):")
            for idx, row in positive_results.iterrows():
                print(f"{row['sequence']} -> 得分: {row['Prediction_Score']:.4f}")
        else:
            print("\n8. 无阳性预测结果")
        
        print(f"\n预测完成! 结果已保存到: {output_file}")
        
    except Exception as e:
        print(f"预测过程中出错: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== 肿瘤抗原预测（使用CNN和新特征集） ===
1. 加载新特征集CNN模型...
加载了 109 个特征索引
2. 加载预测数据...
成功加载 100000 条序列
3. 提取新特征集...
成功加载 20 种氨基酸的物化性质
正在为 100000 个序列提取新特征集...
已处理 1000/100000 个序列
已处理 2000/100000 个序列
已处理 3000/100000 个序列
已处理 4000/100000 个序列
已处理 5000/100000 个序列
已处理 6000/100000 个序列
已处理 7000/100000 个序列
已处理 8000/100000 个序列
已处理 9000/100000 个序列
已处理 10000/100000 个序列
已处理 11000/100000 个序列
已处理 12000/100000 个序列
已处理 13000/100000 个序列
已处理 14000/100000 个序列
已处理 15000/100000 个序列
已处理 16000/100000 个序列
已处理 17000/100000 个序列
已处理 18000/100000 个序列
已处理 19000/100000 个序列
已处理 20000/100000 个序列
已处理 21000/100000 个序列
已处理 22000/100000 个序列
已处理 23000/100000 个序列
已处理 24000/100000 个序列
已处理 25000/100000 个序列
已处理 26000/100000 个序列
已处理 27000/100000 个序列
已处理 28000/100000 个序列
已处理 29000/100000 个序列
已处理 30000/100000 个序列
已处理 31000/100000 个序列
已处理 32000/100000 个序列
已处理 33000/100000 个序列
已处理 34000/100000 个序列
已处理 35000/100000 个序列
已处理 36000/100000 个序列
已处理 37000/100000 个序列
已处理 38000/100000 个序列
已处理 39000/100000 个序列
已处理 40000/100000 个序列
已处理 41000/100000 个序列
已处理 4

GMLAAGGAGL -> 得分: 1.0000
TLCALIRGV -> 得分: 0.9743
PVGPDASPGVAGF -> 得分: 1.0000
NVTSLLKTV -> 得分: 0.6612
PSTGAMGDRLTAVKAAFQSQYKSHF -> 得分: 1.0000
ILLDTIRDEI -> 得分: 1.0000
LRIQEFIEL -> 得分: 0.5257
DQLLVLPDL -> 得分: 0.8836
ILCEHPNQV -> 得分: 0.9914
SVEVRMEFV -> 得分: 0.5725
ALVSGNNTV -> 得分: 0.9703
YLTHTNPRV -> 得分: 0.9432
GPDGRLLRGYNQFAYD -> 得分: 1.0000
RMPEAASPV -> 得分: 0.8574
GLLDVTCKTV -> 得分: 1.0000
ALATALLCL -> 得分: 0.9214
AQIKAVGKL -> 得分: 0.7055
AMSEVEGLEA -> 得分: 1.0000
WLLDVSHLV -> 得分: 0.9943
YNRNGVNAL -> 得分: 0.5272
FLLVSFTWM -> 得分: 0.7978
LLFRGCLIGA -> 得分: 1.0000
SKPVVLPKGP -> 得分: 1.0000
SLMAPSQFA -> 得分: 0.8509
TLLTSSQSL -> 得分: 0.7906
GLVDSSEKV -> 得分: 0.9909
RMPEAAGPV -> 得分: 0.7341
WTYSGQWMY -> 得分: 0.6510
RLQLCRPSL -> 得分: 0.5046
YLVDSLWES -> 得分: 0.9437
HYALLKELLI -> 得分: 1.0000
FLAFTGVAGYI -> 得分: 1.0000
TIKRELAQL -> 得分: 0.8994
YLATALLTL -> 得分: 0.9977
RSPQSVAPPE -> 得分: 1.0000
NLNPDNYQV -> 得分: 0.9184
YNQDAMERV -> 得分: 0.7553
QLAAVRLYV -> 得分: 0.9912
GLMATPAKL -> 得分: 0.9753
GIFWFVLWI -> 得分: 0.9508
YTG

TATQSFLATCINGVCWTV -> 得分: 1.0000
SLVAIAGLMEV -> 得分: 1.0000
ALNEEAGRLL -> 得分: 1.0000
GVFCGVDAV -> 得分: 0.8907
TLLSQSVAL -> 得分: 0.9314
APRGKSGAA -> 得分: 0.7147
NVFENPTMV -> 得分: 0.9147
MQMAYRFNGI -> 得分: 1.0000
SLLLDSPAV -> 得分: 0.9179
HLAKNVVMV -> 得分: 0.9891
KQLIFDSVFSA -> 得分: 1.0000
KLSKDQFAL -> 得分: 0.9008
VLLCRQPCA -> 得分: 0.6690
VLIDYQRNVPAA -> 得分: 1.0000
FLIPGVSPG -> 得分: 0.5541
LIVDAYGKI -> 得分: 0.8622
LAEFMSTYV -> 得分: 0.5013
VLTIVEASV -> 得分: 0.8300
VAIGQPQKA -> 得分: 0.5614
FLVDDLVDSL -> 得分: 1.0000
QLGQENPQL -> 得分: 0.8670
TMKFFLYVL -> 得分: 0.7509
KLEENLPIL -> 得分: 0.9811
LLPAPADKL -> 得分: 0.9014
NLSSAEVVV -> 得分: 0.9236
LELSPAQLL -> 得分: 0.5774
RMPEAAPRV -> 得分: 0.9538
ILAEADNEWVV -> 得分: 1.0000
GLDHYVYKV -> 得分: 0.9938
ALMADMLAHV -> 得分: 1.0000
GLMSHCIAI -> 得分: 0.9559
FLKKYLYEI -> 得分: 0.9661
YVCRKVFQV -> 得分: 0.9564
ALWWGVVTV -> 得分: 0.9987
AGGEPVRSRW -> 得分: 1.0000
YARDETEFYL -> 得分: 1.0000
SLAHALVEI -> 得分: 0.9969
MLYYLKAEI -> 得分: 0.9422
VIAHRLSTV -> 得分: 0.9167
QLMAGVVYL -> 得分: 0.9964
AAPKVSPHV -> 得分:

RLWDNMRKL -> 得分: 0.9955
FLGVQAGKILL -> 得分: 1.0000
FLLSDVAHA -> 得分: 0.9914
EMFGGFFKSV -> 得分: 1.0000
RLPVEVPSV -> 得分: 0.9843
RIIDQSSNV -> 得分: 0.9651
ALYPGPATFL -> 得分: 1.0000
KLSSTNFAM -> 得分: 0.8165
ALAEATVQA -> 得分: 0.9868
AAHNALQYL -> 得分: 0.7065
GLMLFGHPL -> 得分: 0.8949
SLLWWITQV -> 得分: 0.9986
YLFFDLGYL -> 得分: 0.9947
GLYTSIFGV -> 得分: 0.9990
LRIPGGNIY -> 得分: 0.5004
YLSHNGIEV -> 得分: 0.9899
NVDSKGVHA -> 得分: 0.6077
AAFLALGKQG -> 得分: 1.0000
NMINFAPNI -> 得分: 0.8980
LQLDGSLTM -> 得分: 0.8199
NLSLFWKEL -> 得分: 0.7098
ILLGILVFG -> 得分: 0.6039
FAKLEKLTI -> 得分: 0.6372
ALGPQGQRL -> 得分: 0.9379
MADEDALRKI -> 得分: 1.0000
SLLAGIPQRV -> 得分: 1.0000
STAGSAAAMAGAK -> 得分: 1.0000
LLAALKNVQL -> 得分: 1.0000
DEYSLVREL -> 得分: 0.7483
HLYGFPQPRL -> 得分: 1.0000
LLCSDLVKA -> 得分: 0.8600
EVMFALREL -> 得分: 0.5168
WMVMFTPLV -> 得分: 0.9110
TVPERGLLL -> 得分: 0.8385
VYHNHIFWTL -> 得分: 1.0000
NAPAFTQTSY -> 得分: 1.0000
FLLHKQWTL -> 得分: 0.9921
ALDDSVSSSSVHA -> 得分: 1.0000
YLALVKVFN -> 得分: 0.5546
ATSSPGQRV -> 得分: 0.5103
LIAASQSFL -> 得分: 0.71

RMREAAPPV -> 得分: 0.8164
GLWEKSKGDINL -> 得分: 1.0000
YYPIIRKQF -> 得分: 0.6296
GLADKPLFR -> 得分: 0.8311
GVGKSTLLLL -> 得分: 1.0000
FYRVTTEQY -> 得分: 0.5242
PTALPAVPPP -> 得分: 1.0000
AHLEKLLQYC -> 得分: 1.0000
VQKPTTVNV -> 得分: 0.6865
RLVQDTPFL -> 得分: 0.9669
RLDNGTWVL -> 得分: 0.9932
SLADASALVAA -> 得分: 1.0000
LQNYVEEKL -> 得分: 0.7962
KTEVVGISSL -> 得分: 1.0000
SVGGLFWMF -> 得分: 0.5708
TLKIENEEV -> 得分: 0.6912
KTFDWTQFR -> 得分: 0.5747
MFIVVLVQL -> 得分: 0.6690
KVDGAYVAV -> 得分: 0.9708
PNASVYQPLKK -> 得分: 1.0000
YIIPIDEKL -> 得分: 0.9939
ALVAKGLVQA -> 得分: 1.0000
AEELEQRNIL -> 得分: 1.0000
LTAQFEAAV -> 得分: 0.6396
FLGPEIFFHPE -> 得分: 1.0000
TLYESRFQL -> 得分: 0.9897
ITPPQTTPL -> 得分: 0.6268
VLAAANLYA -> 得分: 0.9410
VLLLVLAGV -> 得分: 0.8768
ILDPNQENV -> 得分: 0.9797
QVLLTPVLA -> 得分: 0.5892
KLFIGLISV -> 得分: 0.9974
VVSSVTTKV -> 得分: 0.7663
SVDSVLKRM -> 得分: 0.7252
LYSRQLYVL -> 得分: 0.7186
ISFLSIFLF -> 得分: 0.5485
AQTVQVAEV -> 得分: 0.9201
ILFTEDPRVFKV -> 得分: 1.0000
MIKLKTKPL -> 得分: 0.6953
WLAGDVPAA -> 得分: 0.9621
ALLGDPVFYL -> 得分: 1.00